In [1]:
#Install Packages
!pip install faiss-cpu
!pip install sentence-transformers

   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
    --------------------------------------- 0.3/16.4 MB ? eta -:--:--
   - -------------------------------------- 0.8/16.4 MB 1.4 MB/s eta 0:00:12
   - -------------------------------------- 0.8/16.4 MB 1.4 MB/s eta 0:00:12
   -- ------------------------------------- 1.0/16.4 MB 1.3 MB/s eta 0:00:12
   --- ------------------------------------ 1.3/16.4 MB 1.3 MB/s eta 0:00:12
   ---- ----------------------------------- 1.8/16.4 MB 1.3 MB/s eta 0:00:12
   ----- ---------------------------------- 2.1/16.4 MB 1.3 MB/s eta 0:00:12
   ----- ---------------------------------- 2.4/16.4 MB 1.3 MB/s eta 0:00:11
   ------ --------------------------------- 2.6/16.4 MB 1.4 MB/s eta 0:00:10
   ------- -------------------------------- 2.9/16.4 MB 1.4 MB/s eta 0:00:10
   -------- ---------------


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached sentence_transformers-5.6.0-py3-none-any.whl.metadata (18 kB)
  Using cached huggingface_hub-1.23.0-py3-none-any.whl.metadata (14 kB)
  Using cached torch-2.13.0-cp314-cp314-win_amd64.whl.metadata (39 kB)
  Using cached scikit_learn-1.9.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached filelock-3.29.7-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.5.1-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached sentence_transformers-5.6.0-py3-none-any.whl (596 kB)
   -------------------------


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# import necessary libraries
import pandas as pd
pd.set_option('display.max_colwidth', 100)

In [3]:
df = pd.read_csv("sample_text.csv")
df.shape

(8, 2)

In [4]:
df

,text,category
0,Meditation and yoga can improve mental health,Health
1,"Fruits, whole grains and vegetables helps control blood pressure",Health
2,These are the latest fashion trends for this week,Fashion
3,Vibrant color jeans for male are becoming a trend,Fashion
4,The concert starts at 7 PM tonight,Event
5,Navaratri dandiya program at Expo center in Mumbai this october,Event
6,Exciting vacation destinations for your next trip,Travel
7,Maldives and Srilanka are gaining popularity in terms of low budget vacation places,Travel


Step 1 : Create source embeddings for the text column

In [5]:
from sentence_transformers import SentenceTransformer

C:\Users\addwi\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
encoder = SentenceTransformer("all-mpnet-base-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2006.98it/s]


In [22]:
vectors = encoder.encode(df["text"].tolist())
vectors.shape

(8, 768)

In [24]:
dim = vectors.shape[1]
dim

768

Step 2 : Build a FAISS Index for vectors

In [25]:
import faiss

index = faiss.IndexFlatL2(dim)

Step 3 : Normalize the source vectors (as we are using L2 distance to measure similarity) and add to the index

In [26]:
index.add(vectors)

In [27]:
index

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x00000222C2ECFA70> >

Step 4 : Encode search text using same encorder and normalize the output vector

In [28]:
search_query = "I want to buy a polo t-shirt"
# search_query = "looking for places to visit during the holidays"
# search_query = "An apple a day keeps the doctor away"
vec = encoder.encode(search_query)
vec.shape

(768,)

In [29]:
import numpy as np
svec = np.array(vec).reshape(1,-1)
svec.shape

(1, 768)

Step 5: Search for similar vector in the FAISS index created

In [33]:
distances , I = index.search(svec, k = 2 )

In [34]:
distances

array([[1.3844836, 1.4039092]], dtype=float32)

In [35]:
 I 

array([[3, 2]])

In [36]:
I.tolist()

[[3, 2]]

In [37]:
row_indices = I.tolist()[0]
row_indices

[3, 2]

In [38]:
df.loc[row_indices]

,text,category
3,Vibrant color jeans for male are becoming a trend,Fashion
2,These are the latest fashion trends for this week,Fashion


In [39]:
search_query

'I want to buy a polo t-shirt'